# Colab RAG Ultimate — Single Notebook with Gradio Chat UI

This notebook runs entirely in Google Colab and provides a small Retrieval-Augmented Generation (RAG) demo using:
- `sentence-transformers` for embeddings
- a simple in-memory cosine similarity retriever
- `openai` to generate answers (you provide your API key)
- `gradio` for an in-notebook chat UI
- PDF ingestion for custom documents

Run cells top-to-bottom. Enter your **OpenAI API key** when prompted.


In [ ]:
# Install required libraries
!pip install --upgrade pip --quiet
!pip install sentence-transformers openai gradio pypdf2 scikit-learn --quiet
print('Install finished')

In [ ]:
import os
from getpass import getpass
import openai

# Set your OpenAI API key
if 'OPENAI_API_KEY' not in os.environ:
    key = getpass('Enter your OpenAI API key (it will not be shown): ')
    os.environ['OPENAI_API_KEY'] = key
openai.api_key = os.environ.get('OPENAI_API_KEY')
print('OpenAI key set')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load embedding model
EMBED_MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
print('Loading embedding model...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print('Embedding model loaded')

In [ ]:
# In-memory document store and embeddings
docs = [
    {'id': 'doc1', 'title': 'Haystack overview', 'text': 'Haystack is a framework for building search systems and RAG pipelines.'},
    {'id': 'doc2', 'title': 'RAG concept', 'text': 'Retrieval-augmented generation improves answer accuracy by providing context.'},
    {'id': 'doc3', 'title': 'Colab note', 'text': 'Google Colab is a free environment for running Python notebooks.'}
]

def compute_embeddings(docs_list):
    texts = [d['text'] for d in docs_list]
    embs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    return embs

embeddings = compute_embeddings(docs)
print('Indexed', len(docs), 'documents')

In [ ]:
# Retrieval function
def retrieve(query, top_k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    idxs = np.argsort(sims)[::-1][:top_k]
    results = []
    for i in idxs:
        results.append({'score': float(sims[i]), 'document': docs[i]})
    return results

# Quick test
print(retrieve('What is Haystack?', top_k=2))

In [ ]:
# Build prompt from retrieved docs
def build_prompt(query, retrieved):
    context = ''
    for r in retrieved:
        t = r['document']['text']
        context += '\n- ' + (t if len(t) < 1000 else t[:1000] + '...')
    prompt = f"You are an assistant that answers questions using the provided context.\nContext:\n{context}\n\nQuestion: {query}\nAnswer concisely." 
    return prompt

In [ ]:
# Generate answer using OpenAI GPT-3.5-turbo
def generate_answer_with_openai(query, top_k=3, max_tokens=256, temperature=0.2):
    retrieved = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    messages = [
        {"role": "system", "content": "You are an AI assistant that answers using the provided context."},
        {"role": "user", "content": prompt}
    ]
    resp = openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )
    answer = resp['choices'][0]['message']['content'].strip()
    return {'answer': answer, 'retrieved': retrieved}

In [ ]:
# PDF ingestion helper
from pypdf import PdfReader
def ingest_pdf(file_path, doc_id_prefix='pdf'):
    reader = PdfReader(file_path)
    texts = []
    for p in reader.pages:
        txt = p.extract_text()
        if txt:
            texts.append(txt)
    new_docs = []
    for i, t in enumerate(texts):
        new = {'id': f'{doc_id_prefix}_{len(docs)+i+1}', 'title': os.path.basename(file_path), 'text': t}
        new_docs.append(new)
    if new_docs:
        docs.extend(new_docs)
        global embeddings
        embeddings = np.vstack([embeddings, compute_embeddings(new_docs)])
    return len(new_docs)

In [ ]:
# Gradio chat UI
import gradio as gr
def chat_fn(user_input):
    if not user_input.strip():
        return 'Please ask a question.'
    out = generate_answer_with_openai(user_input, top_k=3)
    answer = out['answer']
    sources = '\n'.join([f"- {r['document']['title']} (score={r['score']:.3f})" for r in out['retrieved']])
    return f"{answer}\n\nSources:\n{sources}"

iface = gr.Interface(fn=chat_fn,
                     inputs=gr.Textbox(lines=2, placeholder='Ask a question...'),
                     outputs=gr.Textbox(),
                     title='Colab RAG — Chat UI',
                     description='Small RAG demo using sentence-transformers + OpenAI')

# Launch in notebook; set share=True for public link
iface.launch(share=False)